In [12]:
import os
folder = r"C:\Users\olivia.jones\OneDrive - West Point\COW YEAR\SE370\Code + Files"
print(os.listdir(folder))

INPUT_PATH = r"C:\Users\olivia.jones\OneDrive - West Point\COW YEAR\SE370\Code + Files\20260420.gkg.csv"
OUTPUT_PATH = r"C:\Users\olivia.jones\OneDrive - West Point\COW YEAR\SE370\Code + Files\course.project.data.clean.part2.csv"

['20260415_clean.csv', '20260415_dnu.csv', '20260415_export.CSV', '20260420.gkg.csv', '911_edges.csv', '911_node_list.csv', 'assignment_2.py', 'assignment_2_hint.ipynb', 'assignment_3.py', 'catch_up.ipynb', 'cb_2018_us_county_20m.cpg', 'cb_2018_us_county_20m.dbf', 'cb_2018_us_county_20m.prj', 'cb_2018_us_county_20m.shp', 'cb_2018_us_county_20m.shp.ea.iso.xml', 'cb_2018_us_county_20m.shp.iso.xml', 'cb_2018_us_county_20m.shx', 'cb_2018_us_county_20m.zip', 'cb_2018_us_county_500k.cpg', 'cb_2018_us_county_500k.dbf', 'cb_2018_us_county_500k.prj', 'cb_2018_us_county_500k.shp', 'cb_2018_us_county_500k.shp.ea.iso.xml', 'cb_2018_us_county_500k.shp.iso.xml', 'cb_2018_us_county_500k.shx', 'cb_2018_us_county_500k.zip', 'cb_2018_us_state_20m.cpg', 'cb_2018_us_state_20m.dbf', 'cb_2018_us_state_20m.prj', 'cb_2018_us_state_20m.shp', 'cb_2018_us_state_20m.shp.ea.iso.xml', 'cb_2018_us_state_20m.shp.iso.xml', 'cb_2018_us_state_20m.shx', 'cb_2018_us_state_20m.zip', 'census_pop_data.csv', 'coding.assignmen

In [11]:
import pandas as pd

#INPUT_PATH = r"C:\Users\olivia.jones\OneDrive - West Point\COW YEAR\SE370\Code + Files\20260420_gkg.csv"
#OUTPUT_PATH = r"C:\Users\olivia.jones\OneDrive - West Point\COW YEAR\SE370\Code + Files\course.project.data.clean.part2.csv"

GKG_COLUMNS = [
    "DATE", "NUMARTS", "COUNTS", "THEMES", "LOCATIONS",
    "PERSONS", "ORGANIZATIONS", "TONE", "CAMEOEVENTIDS",
    "SOURCES", "SOURCEURLS",
]

rows = []

with open(INPUT_PATH, "r", encoding="utf-8", errors="replace") as f:
    for i, raw_line in enumerate(f):
        line = raw_line.strip().strip(",").strip('"')
        if not line:
            continue
        fields = line.split("\t")
        if i == 0 and fields[0] == "DATE":
            continue
        fields = (fields + [""] * 11)[:11]
        rows.append(fields)

df = pd.DataFrame(rows, columns=GKG_COLUMNS)

df["DATE"] = pd.to_datetime(df["DATE"], format="%Y%m%d", errors="coerce")
df["NUMARTS"] = pd.to_numeric(df["NUMARTS"], errors="coerce")

tone_cols = ["Tone", "TonePositive", "ToneNegative", "Polarity",
             "ActivityReferenceDensity", "SelfGroupReferenceDensity"]
tone_split = df["TONE"].str.split(",", expand=True).iloc[:, :6]
tone_split.columns = tone_cols[:tone_split.shape[1]]
for col in tone_split.columns:
    tone_split[col] = pd.to_numeric(tone_split[col], errors="coerce")

tone_pos = df.columns.get_loc("TONE")
df = df.drop(columns=["TONE"])
for j, col in enumerate(tone_split.columns):
    df.insert(tone_pos + j, col, tone_split[col])

df.to_csv(OUTPUT_PATH, index=False)

print(f"Done: {len(df)} rows, {len(df.columns)} columns")
print(df.head(3))

Done: 77562 rows, 16 columns
        DATE  NUMARTS COUNTS  \
0 2026-04-20      1.0          
1 2026-04-20      1.0          
2 2026-04-20      1.0          

                                              THEMES  \
0  TAX_ETHNICITY;TAX_ETHNICITY_CHINESE;TAX_WORLDL...   
1  TAX_WORLDMAMMALS;TAX_WORLDMAMMALS_FOX;TAX_POLI...   
2  EPU_ECONOMY_HISTORIC;TAX_FNCACT;TAX_FNCACT_FOU...   

                                           LOCATIONS  \
0  1#China#CH#CH#35#105#CH;4#Colombo", Western," ...   
1  2#Texas", United States#US#USTX#31.106#-97.647...   
2  4#Dinjan", Assam, India#IN#IN03#27.5#95.1667#-...   

                                             PERSONS  \
0                                           anne aly   
1  jocelyn nungaray;john whitmire;greg abbott;pet...   
2  ajay vannal;dharmesh radadiya;akshaya tritiya;...   

                                       ORGANIZATIONS  Tone  TonePositive  \
0  national asian languages;foreign affairs;chine...   NaN      4.212860   
1  city council